# Variables, modèle et évaluation

Exécuter d'abord `01_exploration.ipynb` et conserver son inventaire dans Drive. Ce notebook entraîne un Random Forest MLlib sur des caractéristiques de couleur et de texture, puis produit les fichiers du dashboard. Aucun résultat n'est prérempli.


In [ ]:
%pip install pyspark==3.5.3 pillow numpy scikit-image scikit-learn pandas


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json

DATA_DIR = '/content/drive/MyDrive/malaria/cell_images'
INVENTORY_DIR = '/content/drive/MyDrive/malaria/outputs/inventory'
RUN_DIR = Path('/content/drive/MyDrive/malaria/outputs') / datetime.now(timezone.utc).strftime('run_%Y%m%d_%H%M%S')

# Option recommandée : CSV path,patient_id avec un identifiant patient vérifié.
PATIENT_MAP_CSV = None
# Activer explicitement uniquement si les identifiants patients sont indisponibles.
ALLOW_IMAGE_SPLIT = False
SEED = 42


## 1. Lire les données validées

Les images sont relues depuis leur dossier, puis reliées à l'inventaire par chemin et hash. Un déplacement ou une modification des fichiers nécessite de régénérer l'inventaire.


In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
spark = SparkSession.builder.master('local[*]').appName('MalariaTraining').getOrCreate()
inventory = spark.read.parquet(INVENTORY_DIR)
binary = (spark.read.format('binaryFile').option('recursiveFileLookup', 'true').load(DATA_DIR)
          .select('path', 'content').withColumn('image_hash', F.sha2('content', 256)))
images = inventory.join(binary, ['path', 'image_hash'], 'inner').cache()
if images.count() != inventory.count():
    raise ValueError("Certaines images ne correspondent plus à l'inventaire. Relancer l'exploration.")
images.groupBy('category').count().show()


## 2. Extraire les caractéristiques

Redimensionnement en 64 × 64 pixels, moyenne et écart-type RGB, histogrammes RGB et histogramme de texture LBP. Ces caractéristiques constituent une baseline ; elles ne remplacent pas un CNN. Tous les pixels sont utilisés, y compris le fond.


In [ ]:
from io import BytesIO
from PIL import Image
import numpy as np
from skimage.feature import local_binary_pattern

FEATURE_NAMES = (
    [f'mean_{c}' for c in 'rgb'] + [f'std_{c}' for c in 'rgb']
    + [f'hist_{c}_{i}' for c in 'rgb' for i in range(8)]
    + [f'lbp_{i}' for i in range(10)]
)

def extract_features(content):
    with Image.open(BytesIO(bytes(content))) as image:
        image = image.convert('RGB').resize((64, 64), Image.Resampling.BILINEAR)
        pixels = np.asarray(image, dtype=np.uint8)
        scaled = pixels.astype(float) / 255.0
        values = list(scaled.mean(axis=(0, 1))) + list(scaled.std(axis=(0, 1)))
        for channel in range(3):
            hist, _ = np.histogram(pixels[:, :, channel], bins=8, range=(0, 256))
            values.extend(hist / hist.sum())
        gray = np.asarray(image.convert('L'))
        lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
        hist, _ = np.histogram(lbp, bins=np.arange(11))
        values.extend(hist / hist.sum())
        return [float(v) for v in values]

feature_udf = F.udf(extract_features, T.ArrayType(T.DoubleType()))
featured = images.withColumn('values', feature_udf('content'))
for i, name in enumerate(FEATURE_NAMES):
    featured = featured.withColumn(name, F.col('values')[i])
featured = featured.drop('content', 'values').cache()
featured.select('category', *FEATURE_NAMES[:6]).show(5)


## 3. Séparer entraînement et test

Le CSV patient doit couvrir chaque image, sans patient vide ou contradictoire. Une répartition déterministe par patient évite qu'un patient apparaisse dans les deux ensembles. Sans ce fichier, il faut activer consciemment `ALLOW_IMAGE_SPLIT` : les scores obtenus auront une limite importante de généralisation.


In [ ]:
if PATIENT_MAP_CSV:
    patient_map = spark.read.option('header', True).csv(PATIENT_MAP_CSV).select('path', 'patient_id')
    if patient_map.filter(F.col('patient_id').isNull() | (F.trim('patient_id') == '')).count():
        raise ValueError('Identifiant patient vide.')
    if patient_map.groupBy('path').count().filter('count > 1').count():
        raise ValueError('Le CSV patient contient plusieurs lignes pour une image.')
    featured = featured.join(patient_map, 'path', 'left')
    if featured.filter(F.col('patient_id').isNull()).count():
        raise ValueError('Le CSV patient ne couvre pas toutes les images.')
    if (featured.groupBy('image_hash').agg(F.countDistinct('patient_id').alias('n'))
            .filter('n > 1').count()):
        raise ValueError('Une image est associée à plusieurs patients.')
    split_key = 'patient_id'
    split_method = 'patient'
elif ALLOW_IMAGE_SPLIT:
    split_key = 'image_hash'
    split_method = 'image_hash — identifiants patients indisponibles'
else:
    raise ValueError('Fournir PATIENT_MAP_CSV ou activer ALLOW_IMAGE_SPLIT après lecture de la limite.')

featured = featured.withColumn('split_bucket', F.pmod(F.xxhash64(split_key, F.lit(SEED)), F.lit(100)))
train = featured.filter('split_bucket < 80')
test = featured.filter('split_bucket >= 80')
for name, frame in [('train', train), ('test', test)]:
    counts = {r.category: r['count'] for r in frame.groupBy('category').count().collect()}
    print(name, counts)
    if set(counts) != {'Parasitized', 'Uninfected'}:
        raise ValueError(f'Les deux catégories doivent être présentes dans {name}.')
if train.select(split_key).intersect(test.select(split_key)).count():
    raise AssertionError('Fuite entre entraînement et test.')


## 4. Entraîner le pipeline

Les labels sont ordonnés alphabétiquement pour fixer Uninfected = 0 et Parasitized = 1. Le pipeline est ajusté uniquement sur l'entraînement. Les paramètres sont fixés avant de consulter les résultats du test.


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.functions import vector_to_array

pipeline = Pipeline(stages=[
    StringIndexer(inputCol='category', outputCol='label', stringOrderType='alphabetDesc'),
    VectorAssembler(inputCols=FEATURE_NAMES, outputCol='features'),
    RandomForestClassifier(labelCol='label', featuresCol='features',
                           numTrees=100, maxDepth=8, seed=SEED)
])
model = pipeline.fit(train)
labels = list(model.stages[0].labels)
assert labels == ['Uninfected', 'Parasitized'], labels
prediction = model.transform(test)
predictions = prediction.select(
    'path', 'category', 'image_hash', 'label', 'prediction',
    F.when(F.col('prediction') == 1, 'Parasitized').otherwise('Uninfected').alias('predicted_category'),
    vector_to_array('probability')[1].alias('probability_parasitized')
).cache()
predictions.select('category', 'predicted_category', 'probability_parasitized').show(10)


## 5. Évaluer et exporter

Le rappel des cellules infectées utilise explicitement Parasitized comme classe positive. La matrice a les catégories réelles en lignes et prédites en colonnes. L'accuracy d'une classe majoritaire sert de référence simple.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
import pandas as pd

result = predictions.toPandas()
y_true = result['category'].eq('Parasitized').astype(int)
y_pred = result['predicted_category'].eq('Parasitized').astype(int)
metrics = {
    'accuracy': float(accuracy_score(y_true, y_pred)),
    'precision_parasitized': float(precision_score(y_true, y_pred, zero_division=0)),
    'recall_parasitized': float(recall_score(y_true, y_pred, zero_division=0)),
    'f1_parasitized': float(f1_score(y_true, y_pred, zero_division=0)),
    'roc_auc': float(roc_auc_score(y_true, result['probability_parasitized'])),
    'n_train': train.count(), 'n_test': len(result),
    'split_method': split_method, 'seed': SEED,
    'train_majority_category': train.groupBy('category').count().orderBy(F.desc('count')).first().category,
    'labels': labels,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
}
metrics['majority_baseline_accuracy'] = float(result['category'].eq(metrics['train_majority_category']).mean())
matrix = confusion_matrix(result['category'], result['predicted_category'], labels=labels)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(pd.DataFrame(matrix, index=labels, columns=labels))

RUN_DIR.mkdir(parents=True, exist_ok=False)
model.write().save(str(RUN_DIR / 'model'))
predictions.write.mode('errorifexists').parquet(str(RUN_DIR / 'predictions_parquet'))
result.to_csv(RUN_DIR / 'predictions.csv', index=False)
(RUN_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding='utf-8')
pd.DataFrame(matrix, index=labels, columns=labels).to_csv(RUN_DIR / 'confusion_matrix.csv')
inventory.groupBy('category').count().toPandas().to_csv(RUN_DIR / 'class_counts.csv', index=False)
importance = pd.DataFrame({'feature': FEATURE_NAMES, 'importance': list(model.stages[-1].featureImportances)})
importance.sort_values('importance', ascending=False).to_csv(RUN_DIR / 'feature_importance.csv', index=False)
print('Dossier exporté :', RUN_DIR)


## 6. Télécharger les fichiers du dashboard

Le ZIP contient uniquement les CSV et métriques, sans images ni modèle. Décompresser son contenu dans `outputs` sur le PC puis lancer `streamlit run app.py`. Le modèle Spark reste dans Drive.


In [ ]:
import zipfile
from google.colab import files
archive = Path('/content/dashboard_results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for filename in ['metrics.json', 'predictions.csv', 'confusion_matrix.csv',
                     'class_counts.csv', 'feature_importance.csv']:
        bundle.write(RUN_DIR / filename, arcname=filename)
files.download(str(archive))
